In [1]:
import unittest

In [2]:
class T_symb_test(unittest.TestCase):
    def setUp(self):
        self.T3=T_symb(3)
        self.T5=T_symb(5)
        self.T7=T_symb(7)
        self.T9=T_symb(9)
        
    def test_init(self):
        self.assertRaises(heis_dim_exception,T_symb,4)
        self.assertRaises(heis_dim_exception,T_symb,0)
        
        self.assertEqual(self.T3.heis_dim,3)
        self.assertEqual(self.T3.basis_strs,['Y','H','E','X','e1','e2','N'])
        self.assertEqual(len(self.T3.basis),7)
        for A in self.T3.basis:
            self.assertEqual(type(A),T_symb_basis_elt)
        
        # short ad_dict test
        X=self.T7.basis[3]
        test_ad_dict={'X':0,'H':-2*self.T7.basis[3],'E':0,'Y':self.T7.basis[1]}
        for i in range(4,len(self.T7.basis)-2):
            test_ad_dict[str(self.T7.basis[i])]=self.T7.basis[i+1]
        test_ad_dict[str(self.T7.basis[len(self.T7.basis)-2])]=0 
        test_ad_dict[str(self.T7.basis[len(self.T7.basis)-1])]=0        
        
        self.assertEqual(X.ad_dict,test_ad_dict)
        self.assertEqual(X.vec_rep,[0]*3+[1]+[0]*(len(self.T7.basis)-4))
        self.assertEqual(X.wght,-1)
        self.assertEqual(X.parent,self.T7)
        self.assertEqual(X.cochain_ad_dicts,{})
        self.assertEqual(X.ext_ad_dicts,{})
    
    def test_sort_basis_tuple(self):
        b0=tuple()
        b1=('X','Y')
        b2=('Y','X')
        b3=('X','Y','e1','e3','H')
        b4=('X','Y','e1','e3','H','banana')
        
        self.assertEqual(self.T9.sort_basis_tuple(b0),(tuple(),1))
        self.assertEqual(self.T7.sort_basis_tuple(b1),(('Y','X'),-1))
        self.assertEqual(self.T7.sort_basis_tuple(b2),(('Y','X'),1))
        self.assertEqual(self.T7.sort_basis_tuple(b3),(('Y','H','X','e1','e3'),1))
        self.assertRaises(ValueError,self.T7.sort_basis_tuple,b4)
        
    def test_elt(self):
        elt0=self.T3.elt([0]*len(self.T3.basis))
        elt1=self.T7.elt([0]*len(self.T7.basis))
        elt2=self.T3.elt([1,-1]+[0]*(len(self.T3.basis)-2))
        elt3=self.T3.elt()
        e2=self.T7.elt([0,0,0,0,0,1]+[0]*(len(self.T7.basis)-6))
        
        self.assertEqual(elt0,elt3)
        self.assertNotEqual(elt0,elt1)
        self.assertEqual(str(elt0),'0')
        self.assertEqual(e2,self.T7.basis[5])
        
        self.assertEqual(elt2,self.T3.basis[0]-self.T3.basis[1])
        self.assertEqual(str(elt2),'Y - H')
    
#     # To do
#     def test_ad(self):
        
#     def test_dual_ad(self):
        

In [3]:
class helpers_test(unittest.TestCase):
    
    def test_str_from_coeff_dict(self):
        # This also tests remove_zeros
        cd0={}
        cd1={'A':0}
        cd2={'A':1}
        cd3={'A':-1}
        cd4={'A':2}
        cd5={'A':1,'B':1}
        cd6={'A':1,'B':-1,'C':0}
        cd7={'A':-1,'B':2,'D':0}
        cd8={'A':2,'B':1}
        cd9={('A','B'):3,('C',):-4}
        cd10={'A':0,('B','C'):0}
        cd11={('A','B','C'):0,'D':-1}
        cd12={('A','B','C'):1,('D','E'):-1}
        cd13={('A','B','C'):-1,('D','E'):1}
        
        self.assertEqual(str_from_coeff_dict(cd0),'0')
        self.assertEqual(str_from_coeff_dict(cd1),'0')
        self.assertEqual(str_from_coeff_dict(cd2),'A')
        self.assertEqual(str_from_coeff_dict(cd3),'- A')
        self.assertEqual(str_from_coeff_dict(cd4),'2*A')
        self.assertEqual(str_from_coeff_dict(cd5),'A + B')
        self.assertEqual(str_from_coeff_dict(cd6),'A - B')
        self.assertEqual(str_from_coeff_dict(cd7),'- A + 2*B')
        self.assertEqual(str_from_coeff_dict(cd8),'2*A + B')
        self.assertEqual(str_from_coeff_dict(cd9),'3*(A,B) + -4*(C)')
        self.assertEqual(str_from_coeff_dict(cd10),'0')
        self.assertEqual(str_from_coeff_dict(cd11),'- D')
        self.assertEqual(str_from_coeff_dict(cd12),'(A,B,C) - (D,E)')
        self.assertEqual(str_from_coeff_dict(cd13),'- (A,B,C) + (D,E)')

In [4]:
class cochain_complex_test(unittest.TestCase):
    def setUp(self):
        self.T3=T_symb(3)
        self.C3=self.T3.cochain_complex
        
        self.T5=T_symb(5)
        self.C5=self.T5.cochain_complex
        
        self.T7=T_symb(7)
        self.C7=self.T7.cochain_complex
        
#     def test_init(self):
#         print(vars(self.C3))
#         print(vars(self.C3.ext_alg))

    def test_wedge_add_mul(self):
        e0=self.T3.ext_alg.elt({})
        e1=self.T3.ext_alg.elt({tuple():1})
        e2=self.T3.ext_alg.elt({('Y','X'):1})
        e3=self.T3.ext_alg.elt({('Y',):1})
        e4=self.T3.ext_alg.elt({('e1',):1})
        
        s0=self.T5.ext_alg.elt({})
        s1=self.T5.ext_alg.elt({tuple():3})
        s2=self.T5.ext_alg.elt({('Y',):2,('E',):-1,('X','Y'):1})
        s3=self.T5.ext_alg.elt({('H','N'):-3,('Y','N'):2})
        s4=self.T5.ext_alg.elt({('Y','H','N'):-6,('E','H','N'):3,('X','Y','H','N'):-3,
                               ('E','Y','N'):-2})
        c0=self.C5.cochain({})
        c1=self.C5.cochain({('Y',):1})
        c2=self.C5.cochain({('X','Y'):-3})
        c3=self.C5.cochain({('X','e1'):1,('Y','e3'):-3})
        
        t0=self.T5.elt()
        t1=self.T5.elt([1]+[0]*(len(self.T5.basis)-1))
        t2=self.T5.elt([-1]+[0]*(len(self.T5.basis)-1))
        t3=self.T5.elt([0]*(len(self.T5.basis)-1)+[1])
        t4=self.T5.elt([1,-2]+[0]*(len(self.T5.basis)-2))
        
        b1=self.T5.basis[0]
        b2=self.T5.basis[1]
        
        # __add__
        
        self.assertEqual(e0+e1,e1)
        self.assertEqual(e2+e0,e2)
        self.assertEqual(e2+e4,self.T3.ext_alg.elt({('Y','X'):1,('e1',):1}))
        self.assertEqual(e3+e3,2*e3)
        self.assertEqual(e3-e3,e0)
        self.assertEqual(-e3,self.T3.ext_alg.elt()-self.T3.ext_alg.elt({('Y',):1}))
        self.assertEqual(-e3,self.T3.ext_alg.elt()+self.T3.ext_alg.elt({('Y',):-1}))
        
        self.assertEqual(c0+c1,c1)
        self.assertEqual(c0-c1,-c1)
        self.assertEqual(c1+2*c2,self.C5.cochain({('Y',):1,('X','Y'):-6}))
        self.assertEqual(4*c2,c2+c2+c2+c2)
        
        self.assertEqual(b1,t1)
        self.assertEqual(b1,b1+t0)
        self.assertEqual(t1,b1+t0)
        self.assertEqual(t1,t1+t0)
        self.assertEqual(t1,t0+b1)
        self.assertEqual(t1,t0+t1)
        
        self.assertEqual(t0-t1,t2)
        self.assertEqual(-t1+t0,t2)
        self.assertEqual(t3+t3,self.T5.elt([0]*(len(self.T5.basis)-1)+[2]))
        self.assertEqual(b1-2*b2,t4)
        
        
        # __mul__
        self.assertEqual(3*e0,e0)
        self.assertEqual(0*e2,e0)
        self.assertEqual(-3*e1,self.T3.ext_alg.elt({tuple():-3}))
        self.assertEqual(3*e2,self.T3.ext_alg.elt({('Y','X'):3}))
        self.assertEqual(e2*3,self.T3.ext_alg.elt({('Y','X'):3}))
        self.assertEqual(e2*0,e0)        
        
        self.assertEqual(4*s0,s0)
        self.assertEqual(0*s0,s0)
        self.assertEqual(0*s4,s0)
        self.assertEqual(-s1,self.T5.ext_alg.elt({tuple():-3}))
        self.assertEqual(s1*(-1),self.T5.ext_alg.elt({tuple():-3}))
        self.assertEqual(s0*4,s0)
        
        self.assertEqual(-c0,c0)
        self.assertEqual(-c1,self.C5.cochain({('Y',):-1}))
        self.assertEqual(5*c2,self.C5.cochain({('X','Y'):-15}))
        self.assertEqual(-2*c3,self.C5.cochain({('X','e1'):-2,('Y','e3'):6}))
        self.assertEqual(c2*5,self.C5.cochain({('X','Y'):-15}))
        
        self.assertEqual(3*t0,t0)
        self.assertEqual(4*t1,self.T5.elt([4]+[0]*(len(self.T5.basis)-1)))
        self.assertEqual(4*b1,self.T5.elt([4]+[0]*(len(self.T5.basis)-1)))    
        self.assertEqual(t1*4,self.T5.elt([4]+[0]*(len(self.T5.basis)-1)))
        self.assertEqual(b1*4,self.T5.elt([4]+[0]*(len(self.T5.basis)-1)))
        self.assertEqual(t4*2,self.T5.elt([2,-4]+[0]*(len(self.T5.basis)-2)))
        
        # wedge
        self.assertEqual(s0.wedge(t0),c0)
        self.assertEqual(s4.wedge(t0),c0)
        self.assertEqual(s0.wedge(t3),c0)
        self.assertEqual(s1.wedge(t1),3*c1)
        self.assertEqual(s1.wedge(b1),3*c1)
        self.assertEqual(s2.wedge(t1),self.C5.cochain(
            {('Y','Y'):2,('E','Y'):-1,('X','Y','Y'):1}))
        
        self.assertEqual(e0.wedge(e1),e0)
        self.assertEqual(e1.wedge(e0),e0)
        self.assertEqual(e1.wedge(e2),e2)
        self.assertEqual(e2.wedge(e1),e2)
        self.assertEqual(e3.wedge(e3),e0)
        self.assertEqual(e2.wedge(e4),self.T3.ext_alg.elt({('Y','X','e1'):1}))
        
        self.assertEqual(s0.wedge(s2),s0)
        self.assertEqual(s2.wedge(s0),s0)
        self.assertEqual(s1.wedge(s2),3*s2)
        self.assertEqual(s2.wedge(s1),3*s2)
        self.assertEqual(s2.wedge(s3),s4)
        
        self.assertEqual(s0.wedge(c1),c0)
        self.assertEqual(s0.wedge(c0),c0)
        self.assertEqual(s1.wedge(c0),c0)
        self.assertEqual(s1.wedge(c1),3*c1)
        self.assertEqual(s1.wedge(c2),3*c2)
        self.assertEqual(s2.wedge(c1),self.C5.cochain({
            ('Y','Y'):2,('E','Y'):-1,('X','Y','Y'):1}))
        self.assertEqual(s2.wedge(c2),self.C5.cochain({
            ('Y','X','Y'):-6,('E','X','Y'):3}))
        
        

In [5]:
%run T_symb.ipynb
%run cochain_complex.ipynb
unittest.main(argv=[''], verbosity=2, exit=False)

test_elt (__main__.T_symb_test) ... ok
test_init (__main__.T_symb_test) ... ok
test_sort_basis_tuple (__main__.T_symb_test) ... ok
test_wedge_add_mul (__main__.cochain_complex_test) ... ok
test_str_from_coeff_dict (__main__.helpers_test) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.015s

OK
